In [1]:
import pandas as pd

# ========= 1. 读取数据 =========
reviews = pd.read_csv(r"D:\rotterdam_airbnb\reviews.csv")
listings = pd.read_csv(r"D:\rotterdam_airbnb\listings.csv")

print("总评论数：", len(reviews))

# ========= 2. 关键词 =========
keywords = [
    "window", "windows", "view", "light",
    "sunlight", "bright", "balcony",
    "glass", "curtain", "privacy",
    "street", "noise"
]

pattern = "|".join(keywords)

# ========= 3. 筛选评论 =========
window_reviews = reviews[
    reviews["comments"].str.contains(pattern, case=False, na=False)
]

print("窗户相关评论数：", len(window_reviews))

# ========= 4. 合并详细房源信息 =========
merged = window_reviews.merge(
    listings,
    left_on="listing_id",
    right_on="id",
    how="left"
)

# ========= 5. 保留关键字段 =========
result = merged[[
    "listing_id",
    "name",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "comments"
]]

# ========= 6. 保存 =========
result.to_csv(
    r"D:\rotterdam_airbnb\rotterdam_window_reviews_detailed.csv",
    index=False
)

print("完成 ✅ 已保存详细窗户评论数据")

总评论数： 63352
窗户相关评论数： 6752
完成 ✅ 已保存详细窗户评论数据


In [2]:
print(result["neighbourhood_cleansed"].value_counts())

neighbourhood_cleansed
Stadsdriehoek        912
Cool                 703
Middelland           618
Noordereiland        367
Hillesluis           319
                    ... 
Lombardijen            1
Spaanse Polder         1
Charlois Zuidrand      1
Pernis                 1
Kralingseveer          1
Name: count, Length: 69, dtype: int64


In [3]:
import folium

m = folium.Map(location=[51.92, 4.48], zoom_start=12)

for _, row in result.iterrows():
    if not pd.isna(row["latitude"]):
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=3,
            popup=row["neighbourhood_cleansed"]
        ).add_to(m)

m.save("rotterdam_window_map.html")

print("地图已生成")

地图已生成


In [4]:
import os
os.getcwd()

'C:\\Users\\86130'

In [12]:
df = pd.read_csv(r"D:\00\爱彼迎民宿评价\rotterdam_airbnb\rotterdam_window_reviews_detailed.csv")
print("读取成功 ✅")

读取成功 ✅


In [13]:
df.head()

,listing_id,name,neighbourhood_cleansed,latitude,longitude,property_type,room_type,comments
0,77592,"Charming, Cozy Rotterdam Center",Middelland,51.91490,4.46322,Entire rental unit,Entire home/apt,We were extremely pleased with our stay at thi...
1,77592,"Charming, Cozy Rotterdam Center",Middelland,51.91490,4.46322,Entire rental unit,Entire home/apt,I stayed in the apartment for 5 days. The loca...
2,77592,"Charming, Cozy Rotterdam Center",Middelland,51.91490,4.46322,Entire rental unit,Entire home/apt,This is a great option if you need privacy and...
3,77592,"Charming, Cozy Rotterdam Center",Middelland,51.91490,4.46322,Entire rental unit,Entire home/apt,It was really nice experience. We've been look...
4,101526,Apartment Dependance Rotterdam,Dijkzigt,51.91246,4.46586,Entire condo,Entire home/apt,i stayed in the beautiful Loe's apartment in n...


In [16]:
# 简单关键词情绪判断
positive_words = ["great", "amazing", "beautiful", "lovely", "excellent", "nice", "perfect", "fantastic"]
negative_words = ["bad", "poor", "dirty", "noisy", "small", "problem", "worst"]

def simple_sentiment(text):
    text = str(text).lower()
    pos = sum(word in text for word in positive_words)
    neg = sum(word in text for word in negative_words)
    
    if pos > neg:
        return "Positive"
    elif neg > pos:
        return "Negative"
    else:
        return "Neutral"

df["sentiment_label"] = df["comments"].apply(simple_sentiment)

print("✅ 情绪分类完成")
df["sentiment_label"].value_counts()

✅ 情绪分类完成


sentiment_label
Positive    5195
Neutral     1309
Negative     248
Name: count, dtype: int64

In [17]:
area_positive_ratio = (
    df.groupby("neighbourhood_cleansed")["sentiment_label"]
      .apply(lambda x: (x == "Positive").mean())
      .sort_values(ascending=False)
)

area_positive_ratio.head(10)

neighbourhood_cleansed
Zuiderpark           1.000000
Vreewijk             1.000000
Prinsenland          1.000000
Charlois Zuidrand    1.000000
De Esch              1.000000
Kleinpolder          1.000000
Dorp                 1.000000
Spangen              0.931507
Noordereiland        0.891008
Katendrecht          0.890625
Name: sentiment_label, dtype: float64

In [18]:
area_negative_ratio = (
    df.groupby("neighbourhood_cleansed")["sentiment_label"]
      .apply(lambda x: (x == "Negative").mean())
      .sort_values(ascending=False)
)

area_negative_ratio.head(10)

neighbourhood_cleansed
Kralingseveer     1.000000
Bloemhof          0.333333
Eemhaven          0.200000
Ommoord           0.129032
Nieuwe Werk       0.117647
Oud Mathenesse    0.111111
Rubroek           0.096774
Struisenburg      0.075472
Bospolder         0.075000
Tussendijken      0.074324
Name: sentiment_label, dtype: float64

In [20]:
import folium

# 先确认有 sentiment_label
print(df.columns)

m = folium.Map(location=[51.92, 4.48], zoom_start=13)

for _, row in df.iterrows():

    if row["sentiment_label"] == "Positive":
        color = "green"
    elif row["sentiment_label"] == "Negative":
        color = "red"
    else:
        color = "blue"

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=3,
        color=color,
        fill=True
    ).add_to(m)

# 保存到你指定路径
m.save(r"D:\00\爱彼迎民宿评价\rotterdam_airbnb\rotterdam_window_map.html")

print("✅ 地图生成成功")

Index(['listing_id', 'name', 'neighbourhood_cleansed', 'latitude', 'longitude',
       'property_type', 'room_type', 'comments', 'sentiment_label'],
      dtype='object')
✅ 地图生成成功


In [21]:
import os
os.getcwd()

'C:\\Users\\86130'

In [22]:
import os
os.path.exists(r"D:\00\爱彼迎民宿评价\rotterdam_airbnb\rotterdam_window_map.html")

True